# Fundamentals of Software Systems (FSS)
**Software Evolution – Part 02 Assignment**

## Submission Guidelines

To correctly complete this assignment you must:

* Carry out the assignment in a team of 2 to 4 students.
* Carry out the assignment with your team only. You are allowed to discuss solutions with other teams, but each team should come up its own personal solution. A strict plagiarism policy is going to be applied to all the artifacts submitted for evaluation.
* As your submission, upload the filled Jupyter Notebook (including outputs) together with the d3 visualization web pages (i.e. upload everything you downloaded including the filled Jupyter Notebook plus your `output.json`)
* The files must be uploaded to OLAT as a single ZIP (`.zip`) file by 2024-12-02 18:00.


## Group Members
* Firstname, Lastname, Immatrikulation Number
* **TO BE FILLED**

## Task Context

In this assigment we will be analyzing the _[Nautilus trader](https://nautilustrader.io/)_ project. The git repository is available here: https://github.com/nautechsystems/nautilus_trader 

All following tasks should be done with the subset of commits from tag `v1.165.0` to tag `v1.206.0`.

## Task 1: Author contributions

In the following, please consider only Rust, Python, and Cython (pyx and pxd) files.

The first task is to get an overview of the author ownership of the Nautilus Trader project. In particular, we want to understand who are the main authors in the system between the two considered tags and what is the amount of their contributions both in absolute terms and in percentages. We also want to investigate if the same patterns apply on the various subsystems.

To this end, you should:
* extract all the contributions between the two tags. If an author committed a file 3 times, then the number of contributions of that author on that file is 3
* sort the authors by total number of contributions, define a threshold and from now on consider only the authors above the threshold
* consider the following subsystems: nautilus_core and all the directories inside nautilus_trader (e.g., accounting, adapters, analysis, ...)
* for the considered authors and the considered subsystems, create a matrix (for example a pandas dataframe) where the columns are the authors, the rows the subsystems, and the value of a cell is the number of contribution of that author on that subsystem
* now comment on the results: is the main author predominant in terms of contributions? how are the contributions distributed among the authors? are the subsystems similar in terms of distribution? To answer these questions compute an additional column that measure the percentage of contributions of the main author with respect to the total.
* redo all the previous steps but instead of counting the number of contributions, for every file in every commit, count the number of lines added. Produce a matrix equivalent to the previous one, using the lines added and comment on the results. Are the results the same if we look at the lines added instead of the number of contributions? For which subsystems are the results different?





In [11]:
import os
import re
import pandas as pd
from git import Repo, NULL_TREE

In [12]:
# Define repository URL and local path
repo_url = 'https://github.com/nautechsystems/nautilus_trader.git'
repo_path = './nautilus_trader'

# Clone the repository if not already cloned
if not os.path.exists(repo_path):
    print('Cloning repository...')
    Repo.clone_from(repo_url, repo_path)
else:
    print('Repository already cloned.')

Repository already cloned.


In [13]:
# Initialize the Repo object
repo = Repo(repo_path)


In [14]:
# Define the tags
start_tag = 'v1.165.0'
end_tag = 'v1.206.0'

# Get commit objects for the tags
start_commit = repo.tags[start_tag].commit
end_commit = repo.tags[end_tag].commit

print(f'Start commit SHA: {start_commit.hexsha}')
print(f'End commit SHA: {end_commit.hexsha}')


Start commit SHA: d54c97abec8d02fbd8abfd930d9922e00dc6cb8f
End commit SHA: 00e23b664505799055279d398dff35d3b9904f0f


In [15]:
# Get list of commits between the two tags
commits = list(repo.iter_commits(f'{start_tag}..{end_tag}'))

print(f'Number of commits between {start_tag} and {end_tag}: {len(commits)}')


Number of commits between v1.165.0 and v1.206.0: 5114


In [16]:
# File extensions of interest
file_extensions = ['.py', '.pyx', '.pxd', '.rs']

# Define subsystems
nautilus_trader_path = os.path.join(repo_path, 'nautilus_trader')
subsystems = ['nautilus_core']
subsystems += [d for d in os.listdir(nautilus_trader_path) if os.path.isdir(os.path.join(nautilus_trader_path, d))]

print('Subsystems:', subsystems)


Subsystems: ['nautilus_core', 'accounting', 'adapters', 'analysis', 'backtest', 'cache', 'common', 'config', 'core', 'data', 'examples', 'execution', 'indicators', 'live', 'model', 'persistence', 'portfolio', 'risk', 'serialization', 'system', 'test_kit', 'trading']


In [18]:
import os
from tqdm import tqdm
import logging
from git import NULL_TREE

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

data = []

logging.info("Starting to process commits...")

for commit in tqdm(commits, desc="Processing commits", unit="commit"):
    author = commit.author.name
    commit_hexsha = commit.hexsha
    parents = commit.parents
    if parents:
        parent = parents[0]
    else:
        parent = None

    try:
        diffs = commit.diff(parent, create_patch=True) if parent else commit.diff(NULL_TREE, create_patch=True)
    except Exception as e:
        logging.error(f"Error generating diff for commit {commit.hexsha}: {e}")
        continue

    for diff in diffs:
        # Handle deleted files
        file_path = diff.b_path if diff.b_path else diff.a_path
        if file_path is None:
            logging.warning(f"Skipping a diff with no file path in commit {commit.hexsha}")
            continue

        _, ext = os.path.splitext(file_path)
        if ext in file_extensions:
            # Determine subsystem
            if file_path.startswith('nautilus_core/'):
                subsystem = 'nautilus_core'
            elif file_path.startswith('nautilus_trader/'):
                parts = file_path.split('/')
                if len(parts) >= 2 and parts[1] in subsystems:
                    subsystem = parts[1]
                else:
                    subsystem = 'other'
            else:
                subsystem = 'other'

            # Count lines added (including deletions)
            try:
                diff_text = diff.diff.decode('utf-8', errors='ignore')
                lines_added = sum(1 for line in diff_text.split('\n') if line.startswith('+') and not line.startswith('+++'))
                lines_deleted = sum(1 for line in diff_text.split('\n') if line.startswith('-') and not line.startswith('---'))
                data.append({
                    'author': author,
                    'subsystem': subsystem,
                    'file_path': file_path,
                    'commit_sha': commit_hexsha,
                    'lines_added': lines_added,
                    'lines_deleted': lines_deleted,
                })
            except Exception as e:
                logging.error(f"Error processing diff for file {file_path} in commit {commit.hexsha}: {e}")
                continue

logging.info("Processing complete. Total commits processed: %d", len(commits))
logging.info("Total data collected: %d entries", len(data))


2024-11-27 02:45:59,464 - INFO - Starting to process commits...
Processing commits: 100%|██████████| 5114/5114 [05:56<00:00, 14.34commit/s]
2024-11-27 02:51:56,050 - INFO - Processing complete. Total commits processed: 5114
2024-11-27 02:51:56,050 - INFO - Total data collected: 26337 entries


In [20]:
df = pd.DataFrame(data)
df.head()


,author,subsystem,file_path,commit_sha,lines_added,lines_deleted
0,Chris Sellers,nautilus_core,nautilus_core/adapters/src/tardis/http/parse.rs,d79fd81bbb7bbfb53249f53e397f1b6e13bc9bc9,1,1
1,Chris Sellers,nautilus_core,nautilus_core/adapters/src/tardis/machine/clie...,d79fd81bbb7bbfb53249f53e397f1b6e13bc9bc9,1,1
2,Chris Sellers,nautilus_core,nautilus_core/adapters/src/tardis/machine/type...,d79fd81bbb7bbfb53249f53e397f1b6e13bc9bc9,1,2
3,Chris Sellers,nautilus_core,nautilus_core/adapters/src/tardis/python/confi...,d79fd81bbb7bbfb53249f53e397f1b6e13bc9bc9,3,3
4,Chris Sellers,nautilus_core,nautilus_core/data/src/engine/tests.rs,d79fd81bbb7bbfb53249f53e397f1b6e13bc9bc9,6,6


Analysis

Count Number of Contributions

In [21]:
# Contributions per author per subsystem
contrib_counts = df.groupby(['author', 'subsystem']).size().reset_index(name='num_contributions')
contrib_counts.head()


,author,subsystem,num_contributions
0,Anurag Roy,adapters,1
1,Anurag Roy,nautilus_core,2
2,Ayush,nautilus_core,10
3,Ayush,other,3
4,Ayush Singh Bhandari,nautilus_core,10


In [23]:
# Define threshold
threshold = 5
active_authors = total_contributions[total_contributions['total_contributions'] >= threshold]['author'].tolist()
print('Active authors:', active_authors)


Active authors: ['Chris Sellers', 'Filip Macek', 'Ishan Bhanuka', 'Pushkar Mishra', 'Brad', 'rsmb7z', 'David Blom', 'faysou', 'Bradley McElroy', 'limx0', 'Benjamin Singleton', 'Reece Kibble', 'Miller Moore', 'ghill2', 'DevRoss', 'graceyangfan', 'Javaid', 'Ayush', 'Ayush Singh Bhandari', 'sunlei', 'Nisayo', 'Dia Kharrat', 'r3k4mn14r']


In [24]:
# Filter the contributions to include only active authors
filtered_contrib_counts = contrib_counts[contrib_counts['author'].isin(active_authors)]


In [25]:
contrib_matrix = filtered_contrib_counts.pivot(index='subsystem', columns='author', values='num_contributions').fillna(0)
contrib_matrix


author,Ayush,Ayush Singh Bhandari,Benjamin Singleton,Brad,Bradley McElroy,Chris Sellers,David Blom,DevRoss,Dia Kharrat,Filip Macek,...,Nisayo,Pushkar Mishra,Reece Kibble,faysou,ghill2,graceyangfan,limx0,r3k4mn14r,rsmb7z,sunlei
subsystem,,,,,,,,,,,,,,,,,,,,,
accounting,0.0,0.0,0.0,6.0,1.0,143.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
adapters,0.0,0.0,60.0,112.0,31.0,2624.0,162.0,15.0,1.0,67.0,...,3.0,0.0,59.0,7.0,0.0,3.0,16.0,0.0,72.0,9.0
analysis,0.0,0.0,0.0,0.0,0.0,91.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0
backtest,0.0,0.0,0.0,9.0,13.0,523.0,4.0,0.0,0.0,2.0,...,0.0,0.0,0.0,4.0,1.0,1.0,4.0,0.0,10.0,0.0
cache,0.0,0.0,0.0,0.0,0.0,244.0,0.0,0.0,0.0,20.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0
common,0.0,0.0,0.0,0.0,5.0,534.0,5.0,0.0,2.0,2.0,...,0.0,0.0,0.0,13.0,1.0,2.0,0.0,0.0,3.0,0.0
config,0.0,0.0,0.0,3.0,4.0,173.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,4.0,0.0
core,0.0,0.0,0.0,1.0,1.0,394.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0
data,0.0,0.0,0.0,1.0,6.0,225.0,5.0,0.0,0.0,2.0,...,0.0,0.0,0.0,22.0,0.0,3.0,1.0,0.0,1.0,0.0


In [26]:
# Identify main author
main_author = total_contributions.iloc[0]['author']
print('Main author:', main_author)

# Compute percentages
contrib_matrix['total_contributions'] = contrib_matrix.sum(axis=1)
contrib_matrix['main_author_percentage'] = (contrib_matrix[main_author] / contrib_matrix['total_contributions']) * 100
contrib_matrix


Main author: Chris Sellers


author,Ayush,Ayush Singh Bhandari,Benjamin Singleton,Brad,Bradley McElroy,Chris Sellers,David Blom,DevRoss,Dia Kharrat,Filip Macek,...,Reece Kibble,faysou,ghill2,graceyangfan,limx0,r3k4mn14r,rsmb7z,sunlei,total_contributions,main_author_percentage
subsystem,,,,,,,,,,,,,,,,,,,,,
accounting,0.0,0.0,0.0,6.0,1.0,143.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,160.0,89.375000
adapters,0.0,0.0,60.0,112.0,31.0,2624.0,162.0,15.0,1.0,67.0,...,59.0,7.0,0.0,3.0,16.0,0.0,72.0,9.0,3346.0,78.421996
analysis,0.0,0.0,0.0,0.0,0.0,91.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,97.0,93.814433
backtest,0.0,0.0,0.0,9.0,13.0,523.0,4.0,0.0,0.0,2.0,...,0.0,4.0,1.0,1.0,4.0,0.0,10.0,0.0,582.0,89.862543
cache,0.0,0.0,0.0,0.0,0.0,244.0,0.0,0.0,0.0,20.0,...,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,269.0,90.706320
common,0.0,0.0,0.0,0.0,5.0,534.0,5.0,0.0,2.0,2.0,...,0.0,13.0,1.0,2.0,0.0,0.0,3.0,0.0,588.0,90.816327
config,0.0,0.0,0.0,3.0,4.0,173.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,4.0,0.0,189.0,91.534392
core,0.0,0.0,0.0,1.0,1.0,394.0,0.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0,424.0,92.924528
data,0.0,0.0,0.0,1.0,6.0,225.0,5.0,0.0,0.0,2.0,...,0.0,22.0,0.0,3.0,1.0,0.0,1.0,0.0,267.0,84.269663


Lines Added Analysis

Sum Lines Added per Author per Subsystem

In [27]:
lines_added_counts = df.groupby(['author', 'subsystem'])['lines_added'].sum().reset_index(name='lines_added')
lines_added_counts.head()


,author,subsystem,lines_added
0,Anurag Roy,adapters,0
1,Anurag Roy,nautilus_core,25
2,Ayush,nautilus_core,383
3,Ayush,other,0
4,Ayush Singh Bhandari,nautilus_core,10


In [28]:
filtered_lines_added_counts = lines_added_counts[lines_added_counts['author'].isin(active_authors)]
lines_added_matrix = filtered_lines_added_counts.pivot(index='subsystem', columns='author', values='lines_added').fillna(0)
lines_added_matrix


author,Ayush,Ayush Singh Bhandari,Benjamin Singleton,Brad,Bradley McElroy,Chris Sellers,David Blom,DevRoss,Dia Kharrat,Filip Macek,...,Nisayo,Pushkar Mishra,Reece Kibble,faysou,ghill2,graceyangfan,limx0,r3k4mn14r,rsmb7z,sunlei
subsystem,,,,,,,,,,,,,,,,,,,,,
accounting,0.0,0.0,0.0,4.0,2.0,568.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
adapters,0.0,0.0,2335.0,2521.0,509.0,24704.0,743.0,39.0,1.0,89.0,...,6.0,0.0,7094.0,44.0,0.0,9.0,73.0,0.0,2209.0,8.0
analysis,0.0,0.0,0.0,0.0,0.0,200.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0
backtest,0.0,0.0,0.0,5.0,39.0,3719.0,10.0,0.0,0.0,5.0,...,0.0,0.0,0.0,9.0,10.0,0.0,5.0,0.0,23.0,0.0
cache,0.0,0.0,0.0,0.0,0.0,1679.0,0.0,0.0,0.0,146.0,...,0.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0,2.0,0.0
common,0.0,0.0,0.0,0.0,7.0,9289.0,1.0,0.0,8.0,15.0,...,0.0,0.0,0.0,14.0,1.0,0.0,0.0,0.0,13.0,0.0
config,0.0,0.0,0.0,9.0,9.0,2026.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,4.0,0.0,3.0,0.0
core,0.0,0.0,0.0,0.0,3.0,4332.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,4.0,0.0
data,0.0,0.0,0.0,1.0,11.0,1996.0,26.0,0.0,0.0,24.0,...,0.0,0.0,0.0,197.0,0.0,0.0,12.0,0.0,0.0,0.0


In [29]:
lines_added_matrix['total_lines_added'] = lines_added_matrix.sum(axis=1)
lines_added_matrix['main_author_percentage'] = (lines_added_matrix[main_author] / lines_added_matrix['total_lines_added']) * 100
lines_added_matrix


author,Ayush,Ayush Singh Bhandari,Benjamin Singleton,Brad,Bradley McElroy,Chris Sellers,David Blom,DevRoss,Dia Kharrat,Filip Macek,...,Reece Kibble,faysou,ghill2,graceyangfan,limx0,r3k4mn14r,rsmb7z,sunlei,total_lines_added,main_author_percentage
subsystem,,,,,,,,,,,,,,,,,,,,,
accounting,0.0,0.0,0.0,4.0,2.0,568.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,608.0,93.421053
adapters,0.0,0.0,2335.0,2521.0,509.0,24704.0,743.0,39.0,1.0,89.0,...,7094.0,44.0,0.0,9.0,73.0,0.0,2209.0,8.0,40742.0,60.635217
analysis,0.0,0.0,0.0,0.0,0.0,200.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,209.0,95.693780
backtest,0.0,0.0,0.0,5.0,39.0,3719.0,10.0,0.0,0.0,5.0,...,0.0,9.0,10.0,0.0,5.0,0.0,23.0,0.0,3960.0,93.914141
cache,0.0,0.0,0.0,0.0,0.0,1679.0,0.0,0.0,0.0,146.0,...,0.0,2.0,0.0,0.0,2.0,0.0,2.0,0.0,1837.0,91.399020
common,0.0,0.0,0.0,0.0,7.0,9289.0,1.0,0.0,8.0,15.0,...,0.0,14.0,1.0,0.0,0.0,0.0,13.0,0.0,9608.0,96.679850
config,0.0,0.0,0.0,9.0,9.0,2026.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,0.0,4.0,0.0,3.0,0.0,2093.0,96.798853
core,0.0,0.0,0.0,0.0,3.0,4332.0,0.0,0.0,0.0,0.0,...,0.0,50.0,0.0,0.0,0.0,0.0,4.0,0.0,4730.0,91.585624
data,0.0,0.0,0.0,1.0,11.0,1996.0,26.0,0.0,0.0,24.0,...,0.0,197.0,0.0,0.0,12.0,0.0,0.0,0.0,2287.0,87.275907


## Task 2: Knowledge loss

We now want to analyze the knowledge loss when the main contributor of the analyzed project would leave. For this we will use the circle packaging layout introduced in the "Code as a Crime Scene" book. This assignment includes the necessary `knowledge_loss.html` file as well as the `d3` folder for all dependencies. Your task is to create the `output.json` file according to the specification below. This file can then be visualized with the files provided.

For showing the visualization, once you have the output as `output.json` you should

* make sure to have the `knowledge_loss.html` file in the same folder
* start a local HTTP server in the same folder (e.g. with python `python3 -m http.server`) to serve the html file (necessary for d3 to work)
* open the served `knowledge_loss.html` and look at the visualization

Based on the visualization, comment on how is the project in terms of project loss and what could happen if the main contributor would leave.


### Output Format for Visualization

* `root` is always the root of the tree
* `size` should be the total number of lines of contribution
* `weight` can be set to the same as `size`
* `ownership` should be set to the percentage of contributions from the main author (e.g. 0.98 for 98% if contributions coming from the main author)

```
{
  "name": "root",
  "children": [
    {
      "name": "test",
      "children": [
        {
          "name": "benchmarking",
          "children": [
            {
              "author_color": "red",
              "size": "4005",
              "name": "t6726-patmat-analysis.scala",
              "weight": 1.0,
              "ownership": 0.9,
              "children": []
            },
            {
              "author_color": "red",
              "size": "55",
              "name": "TreeSetIterator.scala",
              "weight": 0.88,
              "ownership": 0.9,
              "children": []
            }
          ]
        }
      ]
    }
  ]
}
```

### JSON Export

For exporting the data to JSON you can use the following snippet:

```
import json

with open("output.json", "w") as file:
    json.dump(tree, file, indent=4)
```

In [160]:
# your solution here

## Task 3: Code Churn Analysis

The third and last task is to analyze the code churn of the _Nautilus Trader_ project. For this analysis we look at the code churn, meaning the daily change in the total number of lines of the project.

Visualize the code churn over time bucketing the data by day. Remember that you'll need to consider also the days when there are no commits.

Look at the churn trend over time, identify one outlier of your choice, and for it:

* investigate if it was caused by a single or multiple commits (since you are bucketing the data by day)
* find the involved commit(s) and look at the commit message(s)
* find the involved files, and for each file look at the number of lines added and/or deleted as well as the modification type (addition, deletion, modification, renaming)

Based on the above, discuss the potential reasons for the outlier and if it should be a reason for concern.

In [ ]:
# your solution here